# Fine-tune Pretrained MFLSTM on Local Basin

Reads the basin ID from `Input/local.txt`, loads the pretrained model weights (`epoch_30`) and its scaler (`scaler.pickle`), then fine-tunes on that basin and saves the result.

## 1. Imports and Paths

In [ ]:
import os
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

sys.path.append('../_src/_001_aux_functions')
sys.path.append('../_src/_002_readdata')
sys.path.append('../_src/_003_ml_f')

from functions_evaluation import nse
from functions_training import nse_basin_averaged
from utils import Optimizer, create_folder, set_random_seed, upload_to_device, write_report
from camelsh import camelsh as Datasetclass
from mflstm import MFLSTM as modelclass

PROJECT_DIR = Path.cwd()
path_data_default = (PROJECT_DIR / '..' / '..' / 'CAMELSH').resolve()
path_data = Path(os.environ.get('CAMELSH_DATA_DIR', path_data_default)).resolve()
print('Project:', PROJECT_DIR)
print('CAMELSH data:', path_data)

## 2. Read Target Basin from local.txt

In [ ]:
local_basin_path = 'Input/local.txt'
local_basin_id = np.loadtxt(local_basin_path, dtype='str').tolist()
# Wrap scalar string in list so the rest of the code is uniform
if isinstance(local_basin_id, str):
    local_basin_id = [local_basin_id]
print(f'Fine-tuning basin(s) from {local_basin_path}: {local_basin_id}')

## 3. Pretrained Model Paths

In [ ]:
# Weights produced by the full 30-epoch regional training run
pretrained_weights_path = Path('epoch_30')
pretrained_scaler_path  = Path('scaler.pickle')

if not pretrained_weights_path.exists():
    raise FileNotFoundError(f'Pretrained weights not found: {pretrained_weights_path}')
if not pretrained_scaler_path.exists():
    raise FileNotFoundError(f'Pretrained scaler not found: {pretrained_scaler_path}')

print('Pretrained weights :', pretrained_weights_path)
print('Pretrained scaler  :', pretrained_scaler_path)

## 4. Model and Fine-Tuning Configuration

In [ ]:
dynamic_input = {
    '1D': ['CAPE', 'CRainf_frac', 'LWdown', 'PotEvap', 'PSurf', 'Qair', 'Rainf', 'SWdown', 'Tair', 'Wind_E', 'Wind_N'],
    '1h': ['CAPE', 'CRainf_frac', 'LWdown', 'PotEvap', 'PSurf', 'Qair', 'Rainf', 'SWdown', 'Tair', 'Wind_E', 'Wind_N'],
}
target  = ['Q_camelsh_obs_norm']
forcing = ['nldas_hourly']
static_input = [
    'p_mean', 'pet_mean', 'aridity_index', 'p_seasonality', 'frac_snow',
    'high_prec_freq', 'high_prec_dur', 'low_prec_freq', 'low_prec_dur',
    'ele_mt_sav', 'slp_dg_uav', 'ria_ha_usu', 'run_mm_syr', 'gwt_cm_sav',
    'cly_pc_uav', 'slt_pc_uav', 'snd_pc_uav', 'kar_pc_use', 'prm_pc_use',
    'pac_pc_use', 'crp_pc_use', 'for_pc_use', 'urb_pc_use', 'DRAIN_SQKM'
]

# Time periods — use the same as the regional training run
finetune_period  = ['1987-01-01 00:00:00', '2009-12-31 23:00:00']
validation_period = ['2010-01-01 00:00:00', '2015-12-31 23:00:00']
testing_period   = ['2016-01-01 00:00:00', '2022-12-31 23:00:00']
lookback_window  = 0

# Must match the architecture of the pretrained model exactly
model_configuration = {
    'n_dynamic_channels_lstm'  : 10,
    'no_of_layers'             : 1,
    'seq_length'               : 365 * 24,
    'custom_freq_processing'   : {
        '1D': {'n_steps': 351,              'freq_factor': 24},
        '1h': {'n_steps': (365 - 351) * 24, 'freq_factor':  1},
    },
    'predict_last_n'           : 1,
    'unique_prediction_blocks' : True,
    'dynamic_embeddings'       : True,
    'hidden_size'              : 128,     # full model size
    'batch_size_training'      : 16,
    'batch_size_evaluation'    : 256,
    'no_of_epochs'             : 10,      # fine-tuning epochs
    'dropout_rate'             : 0.4,
    # Low LR for fine-tuning — 10× smaller than the original 5e-4
    'learning_rate'            : {1: 5e-5, 5: 1e-5},
    'set_forget_gate'          : 3,
    'validate_every'           : 1,
    'validate_n_random_basins' : -1,
}

seed = 110

## 5. Derived Configuration and Device

In [ ]:
if isinstance(dynamic_input, list):
    model_configuration['dynamic_input_size'] = len(dynamic_input)
elif isinstance(dynamic_input, dict):
    model_configuration['dynamic_input_size'] = {k: len(v) for k, v in dynamic_input.items()}

model_configuration['input_size_lstm'] = (
    model_configuration['n_dynamic_channels_lstm'] + len(static_input)
)
if model_configuration.get('custom_freq_processing') and not model_configuration.get('dynamic_embeddings'):
    model_configuration['input_size_lstm'] += 1

if not model_configuration.get('predict_last_n'):
    model_configuration['predict_last_n'] = 1

if model_configuration.get('predict_last_n', 1) > 1 and not model_configuration.get('unique_prediction_blocks'):
    model_configuration['predict_last_n_evaluation'] = 1
else:
    model_configuration['predict_last_n_evaluation'] = model_configuration.get('predict_last_n', 1)

device = 'cuda:0' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Using device:', device)
print('Torch:', torch.__version__)

## 6. Output Folder

In [ ]:
basin_tag = '_'.join(local_basin_id)
experiment_name    = f'002_finetune_local_{basin_tag}'
path_save_folder   = Path('Results') / experiment_name
create_folder(str(path_save_folder))
print('Output folder:', path_save_folder)

## 7. Load Pretrained Scaler

In [ ]:
with open(pretrained_scaler_path, 'rb') as f:
    pretrained_scaler = pickle.load(f)
print('Scaler keys:', list(pretrained_scaler.keys()))

## 8. Build Fine-Tuning Dataset

We apply the **pretrained scaler** (no refitting) so the model sees normalised values consistent with its learned weights.

In [ ]:
if not (path_data / 'attributes').exists():
    raise FileNotFoundError(f'CAMELSH data folder not found or incomplete: {path_data}')

# Write entity list to a temp file so Datasetclass can read it
finetune_entity_file = path_save_folder / 'finetune_entities.txt'
np.savetxt(finetune_entity_file, local_basin_id, fmt='%s')

start = time.time()
finetune_dataset = Datasetclass(
    dynamic_input=dynamic_input,
    forcing=forcing,
    target=target,
    sequence_length=model_configuration['seq_length'],
    time_period=finetune_period,
    path_data=str(path_data),
    path_entities=str(finetune_entity_file),
    check_NaN=True,
    predict_last_n=model_configuration['predict_last_n'],
    static_input=static_input,
    custom_freq_processing=model_configuration['custom_freq_processing'],
    dynamic_embedding=model_configuration['dynamic_embeddings'],
    unique_prediction_blocks=model_configuration['unique_prediction_blocks'],
    lookback_window=lookback_window,
)
print(f'Fine-tune dataset built in {time.time() - start:.1f}s')
print('Valid samples:', len(finetune_dataset))

## 9. Apply Pretrained Scaler and Build DataLoader

In [ ]:
# Calculate per-basin std (needed for the loss function) but do NOT refit the global scaler
finetune_dataset.calculate_basin_std()
finetune_dataset.scaler = pretrained_scaler
# Save a copy of the scaler into the run folder for reference
with open(path_save_folder / 'scaler.pickle', 'wb') as f:
    pickle.dump(pretrained_scaler, f)

finetune_dataset.standardize_data()

finetune_loader = DataLoader(
    dataset=finetune_dataset,
    batch_size=model_configuration['batch_size_training'],
    shuffle=True,
    drop_last=True,
    collate_fn=finetune_dataset.collate_fn,
)
print('Batches per epoch:', len(finetune_loader))

## 10. Load Pretrained Model Weights

In [ ]:
set_random_seed(seed)
model = modelclass(model_configuration=model_configuration).to(device)

state_dict = torch.load(pretrained_weights_path, map_location=device)
model.load_state_dict(state_dict)
print('Pretrained weights loaded from:', pretrained_weights_path)

optimizer = Optimizer(model=model, model_configuration=model_configuration)

# Verify one forward pass
sample = next(iter(finetune_loader))
sample_device = upload_to_device(sample, device)
with torch.no_grad():
    pred = model(sample_device)
print('Forward pass OK — prediction shape:', pred['y_sim'].shape)

## 11. Optional: Validation Dataset

In [ ]:
validation_dataset = {}
for entity in local_basin_id:
    ds = Datasetclass(
        dynamic_input=dynamic_input,
        forcing=forcing,
        target=target,
        sequence_length=model_configuration['seq_length'],
        time_period=validation_period,
        path_data=str(path_data),
        entity=entity,
        check_NaN=False,
        predict_last_n=model_configuration['predict_last_n'],
        static_input=static_input,
        custom_freq_processing=model_configuration['custom_freq_processing'],
        dynamic_embedding=model_configuration['dynamic_embeddings'],
        unique_prediction_blocks=model_configuration['unique_prediction_blocks'],
        lookback_window=lookback_window,
    )
    ds.scaler = pretrained_scaler
    ds.standardize_data(standardize_output=False)
    validation_dataset[entity] = ds
print('Validation basins:', list(validation_dataset.keys()))

## 12. Fine-Tuning Loop

In [ ]:
training_time  = time.time()
best_validation = -np.inf
best_epoch      = None

for epoch in range(1, model_configuration['no_of_epochs'] + 1):
    epoch_start = time.time()
    total_loss  = []
    model.train()

    for sample in finetune_loader:
        sample = upload_to_device(sample, device)
        optimizer.optimizer.zero_grad()
        pred = model(sample)
        loss = nse_basin_averaged(
            y_sim=pred['y_sim'],
            y_obs=sample['y_obs'],
            per_basin_target_std=sample['basin_std'],
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.optimizer.step()
        total_loss.append(loss.item())

    report = f'Epoch: {epoch:<2} | Loss: {np.mean(total_loss):.4f}'

    if epoch % model_configuration['validate_every'] == 0 and validation_dataset:
        model.eval()
        val_results = {}
        with torch.no_grad():
            for basin, ds in validation_dataset.items():
                loader = DataLoader(
                    dataset=ds,
                    batch_size=model_configuration['batch_size_evaluation'],
                    shuffle=False,
                    drop_last=False,
                    collate_fn=ds.collate_fn,
                )
                frames = []
                for s in loader:
                    s = upload_to_device(s, device)
                    p = model(s)
                    y_sim = (
                        p['y_sim'] * ds.scaler['y_std'].to(device)
                        + ds.scaler['y_mean'].to(device)
                    )
                    frames.append(pd.DataFrame(
                        {
                            'y_obs': s['y_obs'].flatten().cpu().detach(),
                            'y_sim': y_sim[:, -model_configuration['predict_last_n']:, :]
                                     .flatten().cpu().detach(),
                        },
                        index=pd.to_datetime(s['date'].flatten()),
                    ))
                val_results[basin] = (
                    pd.concat(frames, axis=0)
                    if frames
                    else pd.DataFrame(columns=['y_obs', 'y_sim'])
                )

        nse_val = nse(df_results=val_results)
        report += f' | NSE val: {nse_val:.4f}'

        if nse_val > best_validation:
            best_validation = nse_val
            best_epoch      = epoch
            torch.save(model.state_dict(), path_save_folder / 'best_epoch_finetune.pt')

    torch.save(model.state_dict(), path_save_folder / f'epoch_{epoch}')
    torch.save(optimizer.optimizer.state_dict(), path_save_folder / f'optimizer_epoch_{epoch}.pt')
    report += f' | Time: {time.time() - epoch_start:.1f}s | LR: {optimizer.optimizer.param_groups[0]["lr"]:.2e}'
    print(report)
    write_report(str(path_save_folder / 'run_progress.txt'), report)
    optimizer.update_optimizer_lr(epoch=epoch)

print(f'\nTotal fine-tuning time: {time.time() - training_time:.1f}s')
if best_epoch is not None:
    print(f'Best NSE: {best_validation:.4f} at epoch {best_epoch}')

## 13. Test Set Evaluation

In [ ]:
# Load the best fine-tuned checkpoint for evaluation
best_ckpt = path_save_folder / 'best_epoch_finetune.pt'
if best_ckpt.exists():
    model.load_state_dict(torch.load(best_ckpt, map_location=device))
    print('Loaded best fine-tuned checkpoint:', best_ckpt)
else:
    print('No best checkpoint found; using last epoch weights.')

testing_dataset = {}
for entity in local_basin_id:
    ds = Datasetclass(
        dynamic_input=dynamic_input,
        forcing=forcing,
        target=target,
        sequence_length=model_configuration['seq_length'],
        time_period=testing_period,
        path_data=str(path_data),
        entity=entity,
        check_NaN=False,
        predict_last_n=model_configuration['predict_last_n_evaluation'],
        static_input=static_input,
        custom_freq_processing=model_configuration['custom_freq_processing'],
        dynamic_embedding=model_configuration['dynamic_embeddings'],
        unique_prediction_blocks=model_configuration['unique_prediction_blocks'],
        lookback_window=lookback_window,
    )
    ds.scaler = pretrained_scaler
    ds.standardize_data(standardize_output=False)
    testing_dataset[entity] = ds

model.eval()
test_results = {}
with torch.no_grad():
    for basin, ds in testing_dataset.items():
        loader = DataLoader(
            dataset=ds,
            batch_size=model_configuration['batch_size_evaluation'],
            shuffle=False,
            drop_last=False,
            collate_fn=ds.collate_fn,
        )
        frames = []
        for s in loader:
            s = upload_to_device(s, device)
            p = model(s)
            y_sim = (
                p['y_sim'] * ds.scaler['y_std'].to(device)
                + ds.scaler['y_mean'].to(device)
            )
            frames.append(pd.DataFrame(
                {
                    'y_obs': s['y_obs'].flatten().cpu().detach(),
                    'y_sim': y_sim[:, -model_configuration['predict_last_n_evaluation']:, :]
                             .flatten().cpu().detach(),
                },
                index=pd.to_datetime(s['date'].flatten()),
            ))
        test_results[basin] = (
            pd.concat(frames, axis=0)
            if frames
            else pd.DataFrame(columns=['y_obs', 'y_sim'])
        )

with open(path_save_folder / 'test_results_finetune.pickle', 'wb') as f:
    pickle.dump(test_results, f)

nse_vals = nse(df_results=test_results, average=False)
df_NSE   = pd.DataFrame(
    {'basin_id': list(testing_dataset.keys()), 'NSE': np.round(nse_vals, 4)}
).set_index('basin_id')
df_NSE.to_csv(path_save_folder / 'NSE_testing_finetune.csv')
display(df_NSE)